In [13]:
# instalacion de librerias necesarias para el proyecto
!pip install -q langchain langgraph langchain-groq langchain-community langchain-core
!pip install -q pdfplumber pandas

In [14]:
import os
from google.colab import userdata

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
GITHUB_USER = "nikiarrache"
GITHUB_EMAIL = "niki.arrache@gmail.com"
REPO_NAME = "orbitly-agente-ia"

!git config --global user.name "{GITHUB_USER}"
!git config --global user.email "{GITHUB_EMAIL}"
!git clone https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git

print("✅ Repositorio clonado")

Cloning into 'orbitly-agente-ia'...
remote: Enumerating objects: 10, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 10 (delta 0), reused 7 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (10/10), 186.55 KiB | 6.66 MiB/s, done.
✅ Repositorio clonado


In [2]:
import shutil
import os

os.makedirs(f"{REPO_NAME}/documentos", exist_ok=True)

shutil.copy("/content/Orbitly_docs/base_conocimiento_roadmap.pdf", f"{REPO_NAME}/documentos/")
shutil.copy("/content/Orbitly_docs/terminos_de_uso.pdf", f"{REPO_NAME}/documentos/")
shutil.copy("/content/Orbitly_docs/planes_precio.csv", f"{REPO_NAME}/documentos/")
shutil.copy("/content/Orbitly_docs/estado_resultados_2025.csv", f"{REPO_NAME}/documentos/")

print("✅ Documentos copiados al repositorio")

✅ Documentos copiados al repositorio


In [12]:
# Commit 1 enviar documentos a Git
os.chdir(REPO_NAME)

!git add documentos/
!git commit -m "agrego documentos de Orbitly"
!git push https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git

FileNotFoundError: [Errno 2] No such file or directory: 'orbitly-agente-ia'

In [15]:
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
llm = ChatGroq(model="llama-3.3-70b-versatile")

print("✅ Modelo configurado")

✅ Modelo configurado


In [16]:
import pdfplumber
import pandas as pd

def leer_pdf(ruta):
    texto = ""
    with pdfplumber.open(ruta) as pdf:
        for pagina in pdf.pages:
            texto += pagina.extract_text() or ""
    return texto

def leer_csv(ruta):
    df = pd.read_csv(ruta)
    return df.to_string()

def cargar_documentos(carpeta):
    documentos = {}
    for archivo in os.listdir(carpeta):
        ruta = os.path.join(carpeta, archivo)
        if archivo.endswith(".pdf"):
            documentos[archivo] = leer_pdf(ruta)
        elif archivo.endswith(".csv"):
            documentos[archivo] = leer_csv(ruta)
    return documentos

os.chdir(REPO_NAME)
documentos = cargar_documentos("documentos/")
print(f"✅ {len(documentos)} documentos cargados:")
for nombre in documentos:
    print(f"   📄 {nombre}")

✅ 4 documentos cargados:
   📄 terminos_de_uso.pdf
   📄 base_conocimiento_roadmap.pdf
   📄 estado_resultados_2025.csv
   📄 planes_precio.csv


In [17]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

# define la memoria/estado del agente
class EstadoAgente(TypedDict):
    pregunta: str
    respuesta: str

# funcion principal: busca en documentos y responde
def responder(estado):
    pregunta = estado["pregunta"]

    # junta todo el contenido de los documentos
    contexto = ""
    for nombre, contenido in documentos.items():
        contexto += f"\n\n--- {nombre} ---\n{contenido}"

    mensajes = [
        SystemMessage(content="""Eres el asistente virtual interno de Orbitly,
        una plataforma SaaS de gestión de proyectos. Tu trabajo es responder
        preguntas de los colaboradores basándote únicamente en los documentos
        internos de la empresa. Si la respuesta no está en los documentos,
        dilo claramente. Responde siempre en español."""),
        HumanMessage(content=f"""Documentos internos de Orbitly:
        {contexto}

        Pregunta del colaborador: {pregunta}""")
    ]

    respuesta = llm.invoke(mensajes)
    return {"pregunta": pregunta, "respuesta": respuesta.content}

# construir el grafo con LangGraph
grafo = StateGraph(EstadoAgente)
grafo.add_node("responder", responder)
grafo.set_entry_point("responder")
grafo.add_edge("responder", END)
agente = grafo.compile()

print("✅ Agente construido con LangGraph")

✅ Agente construido con LangGraph


In [18]:
# Pruebas Agente
def preguntar(pregunta):
    resultado = agente.invoke({"pregunta": pregunta, "respuesta": ""})
    return resultado["respuesta"]

preguntas = [
    "¿Cuánto cuesta el plan Business de Orbitly?",
    "¿Cuál fue la utilidad neta total de Orbitly en 2025?",
    "¿Qué pasa con mis datos si cancelo mi cuenta?"
]

for p in preguntas:
    print(f"\n🙋 {p}")
    print(f"🤖 {preguntar(p)}")
    print("-" * 60)


🙋 ¿Cuánto cuesta el plan Business de Orbitly?
🤖 Según el archivo `planes_precio.csv`, el plan Business de Orbitly cuesta $39 por mes o $349 por año. Este plan incluye funciones como automatizaciones, acceso a la API, roles y permisos, además de las funciones incluidas en el plan Starter. El plan Business permite tener hasta 25 usuarios incluidos, proyectos ilimitados y 100 GB de almacenamiento. El soporte para este plan es mediante email y chat.
------------------------------------------------------------

🙋 ¿Cuál fue la utilidad neta total de Orbitly en 2025?
🤖 La utilidad neta total de Orbitly en 2025 fue de $230,000. Esto se encuentra en el archivo "estado_resultados_2025.csv", en la fila 13, columna "Utilidad neta". La utilidad neta total es el resultado de sumar las utilidades netas de cada trimestre del año 2025.
------------------------------------------------------------

🙋 ¿Qué pasa con mis datos si cancelo mi cuenta?
🤖 Según la información disponible en el documento "base_co

In [22]:
import shutil

# copia el notebook al repo
shutil.copy("/content/agente_orbitly.ipynb", "/content/orbitly-agente-ia/agente_orbitly.ipynb")

# commit 2
os.chdir("/content/orbitly-agente-ia")
!git add agente_orbitly.ipynb
!git commit -m "construyo agente con LangGraph y funciones de lectura de documentos"
!git push https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git

[main 99e8ab2] construyo agente con LangGraph y funciones de lectura de documentos
 1 file changed, 360 insertions(+)
 create mode 100644 agente_orbitly.ipynb
Enumerating objects: 4, done.
Counting objects: 100% (4/4), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 4.05 KiB | 2.03 MiB/s, done.
Total 3 (delta 0), reused 0 (delta 0), pack-reused 0
To https://github.com/nikiarrache/orbitly-agente-ia.git
   7ccd2a7..99e8ab2  main -> main


In [23]:
#Interfaz con Gradio

import gradio as gr

def chat_agente(pregunta, historial):
    respuesta = preguntar(pregunta)
    return respuesta

interfaz = gr.ChatInterface(
    fn=chat_agente,
    title="🤖 Orbitly AI Assistant",
    description="Hola! Soy el asistente virtual de Orbitly. Puedes preguntarme sobre planes, precios, términos de uso y resultados financieros.",
    examples=[
        "¿Cuánto cuesta el plan Business?",
        "¿Cuál fue la utilidad neta de Orbitly en 2025?",
        "¿Qué pasa con mis datos si cancelo mi cuenta?",
        "¿Qué incluye el plan Enterprise?"
    ],
    theme=gr.themes.Soft()
)

interfaz.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://193a46cf907bde4671.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import shutil

shutil.copy("/content/agente_orbitly.ipynb", "/content/orbitly-agente-ia/agente_orbitly.ipynb")

# copia también la captura de pantalla
shutil.copy("/content/demo_agente.png", "/content/orbitly-agente-ia/demo_agente.png")

os.chdir("/content/orbitly-agente-ia")
!git add agente_orbitly.ipynb demo_agente.png
!git commit -m "agrego interfaz con Gradio y screenshot del agente"
!git push https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git